LOAN DEFAULT PREDICTION...

In [ ]:

import pandas as pd
import numpy as np

df = pd.read_csv("Loan_default.csv")
df.head()


EARLY INSIGHTS OF DEFAULT BORROWERS...

In [ ]:
import matplotlib.pyplot as plt
default_counts = df['Default'].value_counts()
labels = ['No Default', 'Default']
colors = ['teal', 'blue']
explode = (0.05, 0.05)

plt.figure(figsize=(6,6))
plt.pie(default_counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, explode=explode, shadow=True)
plt.title("Loan Default Distribution")
plt.show()

FEATURE ENGINEERING...

In [ ]:
#remove unnecessary columns
df = df.drop('LoanID', axis=1)


# To check the percentage of missing data like NAN
missing_percent = df.isnull().mean()

# Drop the columns having more than 50% missing values
cols_to_drop = missing_percent[missing_percent > 0.5].index
df = df.drop(columns=cols_to_drop)

# Filling the missing values correctly with the median of the column values.
for col in df.columns:
    if col == 'Default':          
        continue
    
    if df[col].dtype in ['int64', 'float64']:   
        df[col] = df[col].fillna(df[col].median())
    else:                                       
        df[col] = df[col].fillna(df[col].mode()[0])

df.head(2)

In [ ]:
df.isnull().sum()

In [ ]:
if {'LoanAmount', 'Income'}.issubset(df.columns):
    df['loan_to_income_ratio'] = df['LoanAmount'] / (df['Income'] + 1) #creating new column(loan_to_income_ratio)

df.head()

In [ ]:

# It convert all object columns(has yes/no or any alphabets) to category and then to numeric codes
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category').cat.codes


df.head()

In [ ]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = df.select_dtypes(include=['float64','int64']).columns
feature_cols = numeric_cols.drop('Default')



df.head()

MODEL TRAINING...

In [ ]:
from sklearn.model_selection import train_test_split
X=df.drop(columns='Default')
y=df["Default"]
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8,random_state=0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print(y_pred)
print(y_test)

In [ ]:
print(y_pred)
print(y_test.values)

GETTING ACCURACY...

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("\n=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ---- Random Forest ----
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("\n=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

print("\n✅ Model training and testing complete!")

In [ ]:

models = ['Logistic Regression', 'Random Forest']
accuracies = [0.885, 0.892]

plt.bar(models, accuracies, color=['blue', 'green'],width=0.4)
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison for Loan Default Prediction')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Taking first row from my dataset
sample_row = X_test[0]

# Show the sample data
print("Sample input data (features):")
print(sample_row)

#predict
pred = rf.predict(sample_row.reshape(1, -1))
print("\nPrediction:", "Default" if pred[0] == 1 else "No Default")